In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
import matplotlib.pyplot as plt
import numpy as np

torch.manual_seed(42)
device = torch.device('cpu')

transform = transforms.Compose([transforms.ToTensor()])
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1000, shuffle=False)

class BaselineCNN(nn.Module):
    def __init__(self):
        super(BaselineCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = x.view(-1, 64 * 7 * 7)
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

def train_model(model, loader, epochs=2, lr=0.001):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    model.train()
    for epoch in range(epochs):
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

print("Training baseline CNN (2 epochs)... done.")
model = BaselineCNN().to(device)
train_model(model, train_loader, epochs=2)

def evaluate_clean(model, loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return (correct / total) * 100

clean_acc = evaluate_clean(model, test_loader)
print(f"Clean test accuracy : {clean_acc:.1f} %")

def fgsm_attack(image, epsilon, data_grad):
    sign_data_grad = data_grad.sign()
    perturbed_image = image + epsilon * sign_data_grad
    perturbed_image = torch.clamp(perturbed_image, 0, 1)
    return perturbed_image

def pgd_attack(model, images, labels, epsilon, alpha, iters=10):
    original_images = images.clone().detach()
    perturbed_images = images.clone().detach() + torch.FloatTensor(*images.shape).uniform_(-epsilon, epsilon).to(device)
    perturbed_images = torch.clamp(perturbed_images, 0, 1)

    for _ in range(iters):
        perturbed_images.requires_grad = True
        outputs = model(perturbed_images)
        loss = nn.CrossEntropyLoss()(outputs, labels)
        model.zero_grad()
        loss.backward()

        data_grad = perturbed_images.grad.data
        perturbed_images = perturbed_images.detach() + alpha * data_grad.sign()
        eta = torch.clamp(perturbed_images - original_images, min=-epsilon, max=epsilon)
        perturbed_images = torch.clamp(original_images + eta, min=0, max=1).detach()

    return perturbed_images

def evaluate_attack(model, loader, attack_type, epsilon, alpha=None):
    model.eval()
    correct = 0
    total = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        if epsilon == 0:
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
            continue

        if attack_type == 'fgsm':
            images.requires_grad = True
            outputs = model(images)
            loss = nn.CrossEntropyLoss()(outputs, labels)
            model.zero_grad()
            loss.backward()
            data_grad = images.grad.data
            perturbed_images = fgsm_attack(images, epsilon, data_grad)
        elif attack_type == 'pgd':
            perturbed_images = pgd_attack(model, images, labels, epsilon, alpha)

        outputs = model(perturbed_images)
        _, predicted = torch.max(outputs.data, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    return (correct / total) * 100

epsilons = [0.00, 0.05, 0.10, 0.20, 0.30]
fgsm_results = []
pgd_results = []

for eps in epsilons:
    if eps == 0:
        fgsm_results.append(clean_acc)
        pgd_results.append(clean_acc)
    else:
        fgsm_results.append(evaluate_attack(model, test_loader, 'fgsm', eps))
        pgd_results.append(evaluate_attack(model, test_loader, 'pgd', eps, alpha=eps/4))

print("EPSILON FGSM_ACC PGD_ACC")
print("-------------------------------")
for i, eps in enumerate(epsilons):
    print(f" {eps:.2f} {fgsm_results[i]:.1f} % {pgd_results[i]:.1f} %")
print("-------------------------------")

sample_loader = DataLoader(Subset(test_dataset, range(5)), batch_size=5, shuffle=False)
for images, labels in sample_loader:
    images, labels = images.to(device), labels.to(device)
    images.requires_grad = True
    outputs = model(images)
    loss = nn.CrossEntropyLoss()(outputs, labels)
    model.zero_grad()
    loss.backward()
    data_grad = images.grad.data
    perturbed_images = fgsm_attack(images, 0.20, data_grad)

    with torch.no_grad():
        clean_preds = model(images).argmax(dim=1).cpu().numpy()
        adv_preds = model(perturbed_images).argmax(dim=1).cpu().numpy()

    fig, axes = plt.subplots(2, 5, figsize=(10, 5))
    for j in range(5):
        axes[0, j].imshow(images[j].squeeze().detach().cpu().numpy(), cmap='gray')
        axes[0, j].set_title(f"Clean: {clean_preds[j]}")
        axes[0, j].axis('off')

        axes[1, j].imshow(perturbed_images[j].squeeze().detach().cpu().numpy(), cmap='gray')
        axes[1, j].set_title(f"Adv: {adv_preds[j]}")
        axes[1, j].axis('off')
    plt.tight_layout()
    plt.savefig('clean_vs_adversarial.png')
    plt.close()

print("Mis-read digits saved : clean_vs_adversarial.png")
print(f"Attack success (eps=0.20): FGSM {100-fgsm_results[3]:.1f} %, PGD {100-pgd_results[3]:.1f} %")

def train_adversarial(model, loader, epochs=2, lr=0.001, epsilon=0.20):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    model.train()
    for epoch in range(epochs):
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            images.requires_grad = True
            outputs = model(images)
            loss = criterion(outputs, labels)
            model.zero_grad()
            loss.backward()
            data_grad = images.grad.data
            perturbed_images = fgsm_attack(images, epsilon, data_grad)

            mixed_images = torch.cat([images[:32], perturbed_images[32:]], dim=0)

            optimizer.zero_grad()
            outputs_mixed = model(mixed_images)
            loss_mixed = criterion(outputs_mixed, labels)
            loss_mixed.backward()
            optimizer.step()

adv_model = BaselineCNN().to(device)
adv_model.load_state_dict(model.state_dict())
train_adversarial(adv_model, train_loader, epochs=2, epsilon=0.20)

adv_clean_acc = evaluate_clean(adv_model, test_loader)
adv_fgsm_acc = evaluate_attack(adv_model, test_loader, 'fgsm', 0.20)
adv_pgd_acc = evaluate_attack(adv_model, test_loader, 'pgd', 0.20, alpha=0.20/4)

print("\nMODEL CLEAN FGSM(0.20) PGD(0.20)")
print("----------------------------------------------------")
print(f"Baseline {clean_acc:.1f} % {fgsm_results[3]:.1f} % {pgd_results[3]:.1f} %")
print(f"Adversarially {adv_clean_acc:.1f} % {adv_fgsm_acc:.1f} % {adv_pgd_acc:.1f} %")
print("trained")
print("----------------------------------------------------")
print(f"Clean accuracy change : {adv_clean_acc - clean_acc:+.1f} %")
print(f"FGSM robustness gain : {adv_fgsm_acc - fgsm_results[3]:+.1f} % PGD gain : {adv_pgd_acc - pgd_results[3]:+.1f} %")

plt.figure(figsize=(8, 5))
plt.plot(epsilons, fgsm_results, 'o-', label='Baseline FGSM')
plt.plot(epsilons, pgd_results, 's-', label='Baseline PGD')
plt.xlabel('Epsilon')
plt.ylabel('Accuracy (%)')
plt.title('Accuracy vs Epsilon')
plt.legend()
plt.grid(True)
plt.savefig('accuracy_vs_epsilon.png')
plt.close()
print("Plot saved : accuracy_vs_epsilon.png")

Training baseline CNN (2 epochs)... done.
Clean test accuracy : 98.8 %
EPSILON FGSM_ACC PGD_ACC
-------------------------------
 0.00 98.8 % 98.8 %
 0.05 95.9 % 95.3 %
 0.10 88.8 % 82.6 %
 0.20 49.9 % 7.1 %
 0.30 11.9 % 0.0 %
-------------------------------
Mis-read digits saved : clean_vs_adversarial.png
Attack success (eps=0.20): FGSM 50.1 %, PGD 92.9 %

MODEL CLEAN FGSM(0.20) PGD(0.20)
----------------------------------------------------
Baseline 98.8 % 49.9 % 7.1 %
Adversarially 98.9 % 96.3 % 7.5 %
trained
----------------------------------------------------
Clean accuracy change : +0.1 %
FGSM robustness gain : +46.4 % PGD gain : +0.4 %
Plot saved : accuracy_vs_epsilon.png
